# Deploy Gemma4 on SageMaker Endpoint using AWS Python API (boto3)

Gemma is a family of open models built by Google DeepMind. Gemma 4 models are multimodal, handling text and image input (with audio supported on small models) and generating text output. This release includes open-weights models in both pre-trained and instruction-tuned variants. Gemma 4 features a context window of up to 256K tokens and maintains multilingual support in over 140 languages.

Featuring both Dense and Mixture-of-Experts (MoE) architectures, Gemma 4 is well-suited for tasks like text generation, coding, and reasoning. The models are available in four distinct sizes: E2B, E4B, 26B A4B, and 31B. Their diverse sizes make them deployable in environments ranging from high-end phones to laptops and servers, democratizing access to state-of-the-art AI.

Gemma 4 introduces key capability and architectural advancements:

- **Reasoning** – All models in the family are designed as highly capable reasoners, with configurable thinking modes.

- **Extended Multimodalities** – Processes Text, Image with variable aspect ratio and resolution support (all models), Video, and Audio (featured natively on the E2B and E4B models).

- **Diverse & Efficient Architectures** – Offers Dense and Mixture-of-Experts (MoE) variants of different sizes for scalable deployment.

- **Optimized for On-Device** – Smaller models are specifically designed for efficient local execution on laptops and mobile devices.

- **Increased Context Window** – The small models feature a 128K context window, while the medium models support 256K.

- **Enhanced Coding & Agentic Capabilities** – Achieves notable improvements in coding benchmarks alongside native function-calling support, powering highly capable autonomous agents.

- **Native System Prompt Support** – Gemma 4 introduces native support for the system role, enabling more structured and controllable conversations.


In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

In [ ]:
import time
import re
import json
import boto3
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name

sm = boto3.client("sagemaker")  # client to intreract with SageMaker
sm_runtime = boto3.client("sagemaker-runtime")  # client to intreract with SageMaker Endpoints

In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    sts = boto3.client('sts')
    response = sts.get_caller_identity()
    assumed_role = response['Arn']
    role = re.sub(r"^(.+)sts::(\d+):assumed-role/(.+?)/.*$", r"\1iam::\2:role/\3", assumed_role)
    return role


def wait_for_endpoint(endpoint_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{endpoint_name}': "
    print(progress)
    
    status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
    
    while status == "Creating" or status == "Updating":
        time.sleep(sleep_time)
        
        status = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"Endpoint: '{endpoint_name}', Status: '{status}'")


def wait_for_ic(ic_name: str, sleep_time: int=60):
    ind = "."
    progress = f"Waiting for '{ic_name}': "
    print(progress)
    
    status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
    
    while status == "Creating" or status == "Updating":
        time.sleep(sleep_time)
        
        status = sm.describe_inference_component(InferenceComponentName=ic_name)["InferenceComponentStatus"]
        
        clear_output(wait=True)
        progress += ind
        print(progress)
  
    print(f"IC: '{ic_name}', Status: '{status}'")

In [ ]:
#
# Overwrite with your role ARN if you are running this notebook outside of SageMaker Studio
#
role = None

if role == None:
    role = get_sagemaker_role()
print(role)

## Container

In [ ]:
instance = {"type": "ml.g6e.12xlarge", "num_gpu": 4}

model_id = "google/gemma-4-31B-it" # or "google/gemma-4-26B-A4B-it"
model_name = f"model-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
timeout = 600
variant_name = "v1"

### vLLM

In [ ]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.19.1-gpu-py312-cu129-ubuntu22.04-sagemaker"

common_env = {
    "VLLM_USE_V2_MODEL_RUNNER": "1"
}
vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "32768",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_TOOL_CALL_PARSER": "gemma4",
    "SM_VLLM_REASONING_PARSER": "gemma4",
}
env = common_env | vllm_env

## Deployment

In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "ModelName": model_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "ContainerStartupHealthCheckTimeoutInSeconds": timeout,
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name, 
                       EndpointConfigName=endpoint_config_name)

_ = wait_for_endpoint(endpoint_name)

## Inference Examples

#### Best Practices

For the best performance, use these configurations and best practices:

1. ***Sampling Parameters***

Use the following standardized sampling configuration across all use cases:
- `temperature=1.0`
- `top_p=0.95`
- `top_k=64`

2. ***Modality order***

For optimal performance with multimodal inputs, place image and/or audio content before the text in your prompt.

---
### Text Generation (non reasoning)

In [80]:
payload={
    "messages": [
        {"role": "user", "content": "Name popular places to visit in London?"}
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 19.52s



London is a massive city with a huge variety of attractions. To make it easier to plan, I have categorized the most popular places to visit based on their appeal:

### 1. The "Must-See" Royal & Historic Landmarks
These are the quintessential London sights that almost every first-time visitor sees:
*   **The Tower of London:** A historic castle and fortress that houses the Crown Jewels.
*   **Westminster Abbey:** The site of royal coronations and weddings.
*   **The Houses of Parliament & Big Ben:** The iconic clock tower and the heart of UK government.
*   **Buckingham Palace:** The King's official residence (don't miss the Changing of the Guard).
*   **Tower Bridge:** The famous bascule bridge (often confused with London Bridge).

### 2. World-Class Museums (Mostly Free!)
London is famous for its incredible, free-entry national museums:
*   **The British Museum:** Dedicated to human history, art, and culture (home to the Rosetta Stone).
*   **The Natural History Museum:** Famous for its dinosaur skeletons and stunning architecture.
*   **The Victoria and Albert (V&A) Museum:** The world's leading museum of art and design.
*   **The National Gallery:** Located in Trafalgar Square, featuring Western European paintings.
*   **Tate Modern:** A huge gallery of international modern and contemporary art.

### 3. Views and Modern Architecture
If you want to see the city from above:
*   **The London Eye:** A giant observation wheel on the South Bank.
*   **The Shard:** The tallest building in Western Europe with an observation deck.
*   **Sky Garden:** A public garden at the top of a skyscraper (free, but requires booking).
*   **The Millennium Bridge:** The pedestrian bridge that leads toward St. Paul's Cathedral.

### 4. Markets and Shopping
For a taste of local culture and retail therapy:
*   **Covent Garden:** Great for street performers, boutiques, and dining.
*   **Camden Market:** Famous for its alternative culture, punk history, and street food.
*   **Borough Market:** The city's most famous food market (perfect for foodies).
*   **Portobello Road Market (Notting Hill):** Famous for antiques and colorful houses.
*   **Oxford Street & Regent Street:** The main hubs for high-street shopping.

### 5. Parks and Leisure
To escape the noise of the city:
*   **Hyde Park:** One of the largest parks, great for rowing boats on the Serpentine.
*   **Regent’s Park:** Beautiful gardens and home to the London Zoo.
*   **St. James’s Park:** The prettiest park, located right next to Buckingham Palace.

### Quick Tips for Visitors:
*   **Transport:** Use a contactless credit card or a mobile wallet (Apple/Google Pay) for the Tube and buses; you don't need to buy a separate "Oyster card" anymore.
*   **Booking:** Many "free" museums and the Sky Garden require a pre-booked time slot online.
*   **Walking:** London is very walkable. Some of the best experiences are found by walking from the South Bank toward the City of London.

-----------------------
{'prompt_tokens': 21, 'total_tokens': 727, 'completion_tokens': 706, 'prompt_tokens_details': None}


---
### Image understanding

<img src="https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png" width="800">

In [81]:
payload={
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": "https://qianwen-res.oss-accelerate.aliyuncs.com/Qwen3.5/demo/RealWorld/RealWorld-04.png"}
                },
                {
                    "type": "text",
                    "text": "Describe this image in detail and tell me where this is"
                }
            ]
        }
    ],
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
display(Markdown(response["choices"][0]["message"]["content"]))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 5.97s



This image shows a large, stylized stone sculpture of a human head and shoulders, appearing as if it is emerging from a rock. The figure wears a golden headband and a golden collar. The sculpture is situated on a balcony or terrace with a sign that reads "Origen" in a stylized cursive font.

In the background, there is a dense urban landscape sprawling across a hillside. The architecture consists largely of low-rise, reddish-brown brick buildings typical of informal settlements or "comunas," with a few taller apartment blocks interspersed. The city is nestled in a valley surrounded by lush green mountains under a bright, partly cloudy sky.

**Location:**
This image was taken in **Medellín, Colombia**. Specifically, it is located in **Comuna 13**, a neighborhood famous for its street art, outdoor escalators, and the "Origen" viewpoint which features this sculpture overlooking the city.

-----------------------
{'prompt_tokens': 292, 'total_tokens': 476, 'completion_tokens': 184, 'prompt_tokens_details': None}


### Test generation (reasoning enabled)

In [82]:
payload={
    "messages": [
        {"role": "user", "content": "What is the derivative of x^3 * ln(x)?"}
    ],
    "chat_template_kwargs": {"enable_thinking": "true"}
}

start_time = time.time()
res = sm_runtime.invoke_endpoint(EndpointName=endpoint_name,
                                 Body=json.dumps(payload),
                                 ContentType="application/json")
response = json.loads(res["Body"].read().decode("utf8"))
end_time = time.time()

print(f"✅ Response time: {end_time-start_time:.2f}s\n")
#print(json.dumps(response, indent=2, default=str))
if response["choices"][0]["message"]["reasoning"]:
    display(Markdown('### Reasoning:\n---'))
    display(Markdown(response["choices"][0]["message"]["reasoning"]))
    #display(Markdown('---'))
display(Markdown('### Content:\n---'))
display(Markdown(response["choices"][0]["message"]["content"]))
display(Markdown('---'))

usage = response["usage"] 
print(f'-----------------------\n{usage}')

✅ Response time: 18.75s



### Reasoning:
---

The user wants to find the derivative of the function $f(x) = x^3 \ln(x)$.

    *   The function is a product of two simpler functions: $u(x) = x^3$ and $v(x) = \ln(x)$.
    *   Therefore, I need to use the **Product Rule**.

    *   The product rule states: $\frac{d}{dx}[u \cdot v] = u'v + uv'$
    *   Let $u = x^3$
    *   Let $v = \ln(x)$

    *   Derivative of $u = x^3$: Using the power rule ($\frac{d}{dx}x^n = nx^{n-1}$), $u' = 3x^2$.
    *   Derivative of $v = \ln(x)$: Using the standard derivative for natural logs, $v' = \frac{1}{x}$.

    *   $\frac{d}{dx}[x^3 \ln(x)] = (3x^2)(\ln(x)) + (x^3)(\frac{1}{x})$

    *   The first term is $3x^2 \ln(x)$.
    *   The second term is $x^3 \cdot \frac{1}{x} = x^2$.
    *   Combine them: $3x^2 \ln(x) + x^2$.

    *   Factor out the common term $x^2$: $x^2(3 \ln(x) + 1)$.

    *   State the formula used.
    *   Show step-by-step differentiation.
    *   Provide the final answer clearly.

### Content:
---

To find the derivative of $x^3 \ln(x)$, you need to use the **Product Rule**.

The product rule states that if you have a function that is the product of two other functions, $u$ and $v$, its derivative is:
$$\frac{d}{dx}[u \cdot v] = u'v + uv'$$

### Step-by-Step Calculation:

**1. Identify the two functions:**
*   $u = x^3$
*   $v = \ln(x)$

**2. Find the derivatives of each part:**
*   $u' = \frac{d}{dx}(x^3) = 3x^2$ (using the power rule)
*   $v' = \frac{d}{dx}(\ln(x)) = \frac{1}{x}$

**3. Plug these into the product rule formula:**
$$\frac{d}{dx}[x^3 \ln(x)] = (3x^2)(\ln(x)) + (x^3)\left(\frac{1}{x}\right)$$

**4. Simplify the expression:**
*   The first term remains $3x^2 \ln(x)$.
*   The second term simplifies: $x^3 \cdot \frac{1}{x} = x^2$.

So, the derivative is:
$$3x^2 \ln(x) + x^2$$

**Final Answer (Factored form):**
You can factor out $x^2$ to make the expression cleaner:
$$x^2(3\ln(x) + 1)$$

---

-----------------------
{'prompt_tokens': 29, 'total_tokens': 761, 'completion_tokens': 732, 'prompt_tokens_details': None}


### Streaming output

In [104]:
#
# Helper function for streamin invocation
#
import io
import json
import time
import boto3
from IPython.display import clear_output

class LineIterator:
    def __init__(self, stream):
        self.byte_iterator = iter(stream)
        self.buffer = io.BytesIO()
        self.read_pos = 0

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            self.buffer.seek(self.read_pos)
            line = self.buffer.readline()
            if line and line[-1] == ord("\n"):
                self.read_pos += len(line)
                return line[:-1]
            try:
                chunk = next(self.byte_iterator)
            except StopIteration:
                if self.read_pos < self.buffer.getbuffer().nbytes:
                    continue
                raise
            if "PayloadPart" not in chunk:
                print("Unknown event type:" + chunk)
                continue
            self.buffer.seek(0, io.SEEK_END)
            self.buffer.write(chunk["PayloadPart"]["Bytes"])

def stream_response(endpoint_name, inputs, max_tokens=8189, temperature=0.7, top_p=0.9):    
    body = {
      "messages": [
            {"role": "user", "content": [{"type": "text", "text": inputs}]}
        ],
        "max_tokens": max_tokens,
        "temperature": temperature,
        "top_p": top_p,
        "stream": True,
        "chat_template_kwargs": {"enable_thinking": "true"}
    }

    resp = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName = endpoint_name,
        Body = json.dumps(body),
        ContentType = "application/json",
    )

    event_stream = resp["Body"]
    start_json = b"{"
    full_response = ""
    start_time = time.time()
    token_count = 0

    for line in LineIterator(event_stream):
        if line != b"" and start_json in line:
            data = json.loads(line[line.find(start_json):].decode("utf-8"))
            delta = data['choices'][0]['delta']
            if "reasoning" in delta:
                token_text = delta.get('reasoning', '')
            else:
                token_text = delta.get('content', '')          
            
            full_response += token_text
            token_count += 1

            # Calculate tokens per second
            elapsed_time = time.time() - start_time
            tps = token_count / elapsed_time if elapsed_time > 0 else 0

            # Clear the output and reprint everything
            clear_output(wait=True)
            display(Markdown(full_response))

    print(f"\nTokens per Second: {tps:.2f}\n")
    
    return full_response

In [105]:
#inputs = "What is greater 9.11 or 9.8?"
inputs = "Solve this problem step by step: What is 15% of 240?"
output = stream_response(endpoint_name, inputs, max_tokens=8000)

The user wants to find 15% of 240.

    *   Number: 240
    *   Percentage: 15%

    *   *Method 1: Multiplication by decimal* (15% = 0.15)
    *   *Method 2: Multiplication by fraction* (15% = 15/100)
    *   *Method 3: Breaking it down* (10% + 5%)

    *   *Method 1 (Decimal):*
        $240 \times 0.15$
        $240 \times 0.1 = 24$
        $240 \times 0.05 = 12$
        $24 + 12 = 36$

    *   *Method 2 (Fraction):*
        $(15/100) \times 240$
        $(3/20) \times 240$
        $240 \div 20 = 12$
        $3 \times 12 = 36$

    *   *Method 3 (Breakdown):*
        10% of 240 = 24
        5% is half of 10%, so 5% of 240 = 12
        15% = 24 + 12 = 36

    *   Step 1: Convert the percentage to a decimal or fraction.
    *   Step 2: Multiply by the number.
    *   Step 3: State the final answer.To find 15% of 240, you can follow these steps:

**Step 1: Convert the percentage to a decimal.**
To turn a percentage into a decimal, divide it by 100:
$15 \div 100 = 0.15$

**Step 2: Multiply the decimal by the number.**
Multiply 0.15 by 240:
$240 \times 0.15 = 36$

***

**Alternative Method (Mental Math):**
If you don't have a calculator, you can break the percentage down into easier parts:
1. **Find 10% of 240:** (Move the decimal point one place to the left)
   $240 \times 0.10 = 24$
2. **Find 5% of 240:** (5% is half of 10%)
   $24 \div 2 = 12$
3. **Add them together:**
   $24 + 12 = 36$

**Final Answer:**
15% of 240 is **36**.


Tokens per Second: 39.00



## Cleanup

In [106]:
_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)